# What CLIP actually is

CLIP (Contrastive Language-Image Pre-training) is two separate encoders trained together:

An image encoder (a ViT or ResNet) that turns an image into a vector
A text encoder (a Transformer) that turns text into a vector

They're trained on ~400M (image, caption) pairs scraped from the internet, with a contrastive loss: for each batch, the correct image-caption pairs should have high cosine similarity, and every mismatched pair in that batch should have low similarity. Over millions of examples, this forces both encoders to project into a shared embedding space — even though the two encoders have completely different architectures internally, their outputs live in the same coordinate system.

That's the property you actually care about for MedRAG: once you have an image embedding and a text embedding in the same space, cosine similarity between them is meaningful. A text query like "chest X-ray showing pneumonia" can be compared directly against a database of image vectors — no separate "caption this image" step needed.

One thing to be upfront about, since it matters for a medical RAG system: CLIP was trained on general internet images, not medical imaging. It'll do fine on your WHO images (mostly diagrams, charts, algorithm flowcharts, not radiology scans), but if you ever added actual clinical images (X-rays, histology slides), general CLIP would be a weak choice — there are medical-domain variants (BiomedCLIP, PMC-CLIP) for that. Worth a one-line note in your report; not a blocker for what you're doing now.

# Load and Inspect WHO Image Metadata

In [3]:
import sys, os
from medrag.ingestion.storage import load_who_images

backend_path = os.path.abspath(os.path.join("..", "backend"))
sys.path.insert(0, backend_path)


WHO_TOPICS = [
    "tuberculosis", "hypertension", "diabetes", "obesity", "asthma", "copd",
    "coronary artery disease", "heart failure", "stroke", "hyperlipidemia",
    "pneumonia", "covid-19", "malaria", "hiv aids", "hepatitis b", "hepatitis c",
    "dengue fever", "typhoid", "depression", "anxiety disorder", "epilepsy",
    "malnutrition", "anemia in pregnancy", "breast cancer",
]

all_images = []
for topic in WHO_TOPICS:
    images = load_who_images(topic, output_dir="../data/images/who")
    all_images.extend(images)
    print(f"{topic}: {len(images)} images")

print()
print(f"Total image entries across all topic files: {len(all_images)}")

tuberculosis: 0 images
hypertension: 9 images
diabetes: 2 images
obesity: 0 images
asthma: 3 images
copd: 3 images
coronary artery disease: 3 images
heart failure: 3 images
stroke: 3 images
hyperlipidemia: 3 images
pneumonia: 4 images
covid-19: 4 images
malaria: 4 images
hiv aids: 0 images
hepatitis b: 8 images
hepatitis c: 4 images
dengue fever: 21 images
typhoid: 2 images
depression: 6 images
anxiety disorder: 6 images
epilepsy: 6 images
malnutrition: 4 images
anemia in pregnancy: 2 images
breast cancer: 0 images

Total image entries across all topic files: 100


# Deduplicate Images by Content Hash, Grouped by Topics

In [4]:
import hashlib
from pathlib import Path
from collections import defaultdict

def compute_file_hash(filepath):
    with open(filepath, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


IMAGE_DIR = "../data/images/who"

hash_to_topics = defaultdict(list)
hash_to_image_record = {}

for topic in WHO_TOPICS:
    images = load_who_images(topic, output_dir=IMAGE_DIR)
    for img in images:
        filepath = Path(IMAGE_DIR) / img.filename
        if not filepath.exists():
            continue
        file_hash = compute_file_hash(filepath)
        hash_to_topics[file_hash].append(topic)
        if file_hash not in hash_to_image_record:
            hash_to_image_record[file_hash] = img

print(f"Total image entries: {sum(len(load_who_images(t, output_dir=IMAGE_DIR)) for t in WHO_TOPICS)}")
print(f"Unique images (by content hash): {len(hash_to_image_record)}")
print()
print("Examples of shared images:")
for h, topics in list(hash_to_topics.items())[:5]:
    if len(topics) > 1:
        print(f"  {hash_to_image_record[h].filename}: {topics}")

Total image entries: 100
Unique images (by content hash): 75

Examples of shared images:


# List All Shared Images

In [5]:
shared_count = 0
for h, topics in hash_to_topics.items():
    if len(topics) > 1:
        shared_count += 1
        print(f"  {hash_to_image_record[h].filename}: {topics}")

print()
print(f"Number of images shared across topics: {shared_count}")

# Also check for any images that failed to load (missing files), which could explain the off-by-one
missing_count = 0
for topic in WHO_TOPICS:
    images = load_who_images(topic, output_dir=IMAGE_DIR)
    for img in images:
        filepath = Path(IMAGE_DIR) / img.filename
        if not filepath.exists():
            missing_count += 1
            print(f"  MISSING FILE: {img.filename} (referenced under topic '{topic}')")

print(f"Missing files: {missing_count}")

  asthma_page9_img0.png: ['asthma', 'copd']
  asthma_page10_img1.png: ['asthma', 'copd']
  asthma_page78_img2.png: ['asthma', 'copd']
  coronary_artery_disease_page12_img0.png: ['coronary artery disease', 'heart failure', 'stroke', 'hyperlipidemia']
  coronary_artery_disease_page13_img1.png: ['coronary artery disease', 'heart failure', 'stroke', 'hyperlipidemia']
  coronary_artery_disease_page17_img2.png: ['coronary artery disease', 'heart failure', 'stroke', 'hyperlipidemia']
  pneumonia_page0_img0.png: ['pneumonia', 'malnutrition']
  depression_page19_img0.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page55_img1.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page70_img2.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page18_img3.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page21_img4.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page22_img5.png: ['depression', 'anxiety disorder', 'epilepsy'

# Exclude Cover-Page-Position Images From Cross-Topic Merging

In [6]:
import re

COVER_PAGE_PATTERN = re.compile(r"_page0_img0\.png$")

hash_to_topics = defaultdict(list)
hash_to_image_record = {}

for topic in WHO_TOPICS:
    images = load_who_images(topic, output_dir=IMAGE_DIR)
    for img in images:
        filepath = Path(IMAGE_DIR) / img.filename
        if not filepath.exists():
            continue

        if COVER_PAGE_PATTERN.search(img.filename):
            # Cover-page-position images are never merged across topics,
            # even if their content hash matches another document's cover
            # graphic (a shared PDF template banner, not real content) -
            # use a per-topic-unique key instead of the content hash
            key = f"{img.filename}_{topic}"
        else:
            key = compute_file_hash(filepath)

        hash_to_topics[key].append(topic)
        if key not in hash_to_image_record:
            hash_to_image_record[key] = img

print(f"Unique images (with cover-page exclusion rule): {len(hash_to_image_record)}")
print()
print("Shared images (excluding cover-page graphics):")
for h, topics in hash_to_topics.items():
    if len(topics) > 1:
        print(f"  {hash_to_image_record[h].filename}: {topics}")

Unique images (with cover-page exclusion rule): 76

Shared images (excluding cover-page graphics):
  asthma_page9_img0.png: ['asthma', 'copd']
  asthma_page10_img1.png: ['asthma', 'copd']
  asthma_page78_img2.png: ['asthma', 'copd']
  coronary_artery_disease_page12_img0.png: ['coronary artery disease', 'heart failure', 'stroke', 'hyperlipidemia']
  coronary_artery_disease_page13_img1.png: ['coronary artery disease', 'heart failure', 'stroke', 'hyperlipidemia']
  coronary_artery_disease_page17_img2.png: ['coronary artery disease', 'heart failure', 'stroke', 'hyperlipidemia']
  depression_page19_img0.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page55_img1.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page70_img2.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page18_img3.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page21_img4.png: ['depression', 'anxiety disorder', 'epilepsy']
  depression_page22_img5.png: ['de

# Load CLIP and Embed One Real Image

In [1]:
import open_clip
import torch
from PIL import Image

model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
model.eval()

print("Model loaded successfully")
print("Preprocessing pipeline:", preprocess)

c:\Users\DELL\Desktop\medrag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\DELL\Desktop\medrag\venv\Lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Model loaded successfully
Preprocessing pipeline: Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    <function _convert_to_rgb at 0x000001A073466A20>
    ToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)


# Embed One Real Image

In [7]:
sample_image_record = hash_to_image_record[list(hash_to_image_record.keys())[0]]
sample_path = Path(IMAGE_DIR) / sample_image_record.filename

img = Image.open(sample_path)
img_tensor = preprocess(img).unsqueeze(0)  # add batch dimension

with torch.no_grad():
    image_features = model.encode_image(img_tensor)

print("Sample image:", sample_image_record.filename)
print("Embedding shape:", image_features.shape)
print("First 5 values:", image_features[0][:5])

Sample image: hypertension_page2_img0.png
Embedding shape: torch.Size([1, 512])
First 5 values: tensor([ 0.4218,  0.0475, -0.2023,  0.2777,  0.3910])


# Similarity Sanity Check

In [8]:
def embed_image(filepath):
    img = Image.open(filepath)
    img_tensor = preprocess(img).unsqueeze(0)
    with torch.no_grad():
        features = model.encode_image(img_tensor)
    return features[0]

def cosine_similarity(a, b):
    return torch.nn.functional.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

# Two images from the SAME shared document (asthma/copd) - expect higher similarity
asthma_img1 = embed_image(Path(IMAGE_DIR) / "asthma_page9_img0.png")
asthma_img2 = embed_image(Path(IMAGE_DIR) / "asthma_page10_img1.png")

# One image from a totally different document (hypertension) - expect lower similarity
hypertension_img = embed_image(Path(IMAGE_DIR) / "hypertension_page2_img0.png")

sim_same_doc = cosine_similarity(asthma_img1, asthma_img2)
sim_diff_doc = cosine_similarity(asthma_img1, hypertension_img)

print(f"Asthma image 1 vs Asthma image 2 (same document): {sim_same_doc:.4f}")
print(f"Asthma image 1 vs Hypertension image (different document): {sim_diff_doc:.4f}")

Asthma image 1 vs Asthma image 2 (same document): 0.7346
Asthma image 1 vs Hypertension image (different document): 0.5482


# Build the Full Batch Embedding Function

In [9]:
def embed_images_batch(image_records, image_dir):
    """
    Embed a list of unique image records (from the dedup step), returning
    (successfully_embedded_records, embeddings_array).
    """
    embeddings = []
    successful_records = []

    for record in image_records:
        filepath = Path(image_dir) / record.filename
        try:
            vec = embed_image(filepath)
            embeddings.append(vec.numpy())
            successful_records.append(record)
        except Exception as e:
            print(f"  Failed to embed {record.filename}: {e}")

    return successful_records, np.array(embeddings)


import numpy as np

# Test on all 76 unique images
unique_records = list(hash_to_image_record.values())
successful_records, image_embeddings = embed_images_batch(unique_records, IMAGE_DIR)

print(f"Successfully embedded: {len(successful_records)} / {len(unique_records)}")
print(f"Embeddings array shape: {image_embeddings.shape}")

Successfully embedded: 76 / 76
Embeddings array shape: (76, 512)


# Attach Topics and Save

In [10]:
import json

def save_image_embeddings(records, embeddings, topics_map, output_dir="../data/processed/embeddings"):
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    npy_path = Path(output_dir) / "who_images_embeddings.npy"
    np.save(npy_path, embeddings)

    index_path = Path(output_dir) / "who_images_index.jsonl"
    with open(index_path, "w", encoding="utf-8") as f:
        for record, key in zip(records, topics_map.keys()):
            pass  # will fix mapping below

    return npy_path, index_path


# Build the record -> topics mapping directly from what we already computed
key_by_filename = {v.filename: k for k, v in hash_to_image_record.items()}

index_records = []
for record in successful_records:
    key = key_by_filename[record.filename]
    index_records.append({
        "filename": record.filename,
        "topics": hash_to_topics[key],
        "page_number": record.page_number,
        "image_type": record.image_type,
    })

npy_path = Path("../data/processed/embeddings/who_images_embeddings.npy")
npy_path.parent.mkdir(parents=True, exist_ok=True)
np.save(npy_path, image_embeddings)

index_path = Path("../data/processed/embeddings/who_images_index.jsonl")
with open(index_path, "w", encoding="utf-8") as f:
    for rec in index_records:
        f.write(json.dumps(rec) + "\n")

print("Saved to:", npy_path.resolve())
print("Index saved to:", index_path.resolve())
print()
print("Sample index entry:", index_records[0])

Saved to: C:\Users\DELL\Desktop\medrag\data\processed\embeddings\who_images_embeddings.npy
Index saved to: C:\Users\DELL\Desktop\medrag\data\processed\embeddings\who_images_index.jsonl

Sample index entry: {'filename': 'hypertension_page2_img0.png', 'topics': ['hypertension'], 'page_number': 2, 'image_type': 'embedded'}
